In [1]:
from pyscf import gto, scf, cc

a = 4 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = '6-31g'
# for nc in nc_list:
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis=basis, unit='B', spin=0, verbose=4)
mol.build()

mf = scf.UHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()[0]

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-14-generic', version='#14~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jan 15 15:52:10 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Mon Mar  2 11:45:12 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    

np.float64(-0.009934679377422505)

In [4]:
# example for PT2

options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 1,
            'dt':0.005,
            'seed': 2,
            'walker_type': 'uhf',
            'trial': 'ustoccsd2',
            'nslater': 100,
            'free_projection':False,
            'fp_abs': False,
            'group': False,
            'ad_mode': None,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted.mixed_wave import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (1, 1)
# Number of basis functions: 4
# Number of Cholesky vectors: 9
#


In [5]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
prop_data["e_estimate"] = e_init
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# Decomposing Unrestricted T2 amplitudes
# Throw 0 vectors in T2 deomposition
# SVD cutoff = 1.00e-08 | error = 1.60e-15
# number of T2 decomposition vectors 6
# norb: 4
# nelec: (1, 1)
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 1
# dt: 0.005
# seed: 2
# walker_type: uhf
# trial: ustoccsd2
# nslater: 100
# free_projection: False
# fp_abs: False
# group: False
# use_gpu: False
# n_exp_terms: 6
# n_batch: 1
# max_error: 0.001
-0.999550687941519
-3.5493297190214435e-10


In [9]:
trial.nslater

100

In [8]:
wave_data['tau'][0].shape

(6, 1, 3)

In [10]:
xtaus = trial.get_xtaus(prop_data, wave_data, prop)

In [12]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
slater_up, slater_dn = wave_data['mo_ta'], wave_data['mo_tb']
trial._calc_energy_slater(walker_up, walker_dn, slater_up, slater_dn, ham_data)

(Array(1.+0.j, dtype=complex128), Array(-1.00012608+0.j, dtype=complex128))

In [16]:
t1 = [mycc.t1[0], mycc.t1[1]]
t2 = [mycc.t2[0]*0, mycc.t2[1]*0, mycc.t2[2]*0]
print(mf.e_tot + mycc.energy(t1, t2))

-1.000126073676908


In [22]:
xtau = [xtaus[0][0,0,:,:], xtaus[1][0,0,:,:]]
wave_data['mo_ta'], wave_data['mo_tb'] = walker_up, walker_dn
o_ex, e_ex = trial._calc_energy_exp_xtau(walker_up, walker_dn, ham_data, wave_data, xtau)
o_ci, e_ci = trial._calc_energy_cisd_xtau(walker_up, walker_dn, ham_data, wave_data, xtau)
print(o_ex, e_ex)
print(o_ci, e_ci)

(1+0j) (-1.0255353211948717+0.003671668913551171j)
(1+0j) (-1.0533154467999484+0.011953383176096878j)


In [27]:
xtau_aa = jnp.einsum('ia,jb->iajb',xtau[0],xtau[0]) / 2
xtau_ab = jnp.einsum('ia,jb->iajb',xtau[0],xtau[1]) / 2
xtau_bb = jnp.einsum('ia,jb->iajb',xtau[1],xtau[1]) / 2
xtau2 = [xtau_aa, xtau_ab, xtau_bb]
print(mf.e_tot + mycc.energy(xtau, xtau2))


WARN: Non-zero imaginary part found in UCCSD energy (-0.02640806828543728-0.005538125931082027j)

-1.0259587558720233


In [30]:
t1 = mycc.t1
o_ci, e_ci = trial._calc_energy_cisd_xtau(walker_up, walker_dn, ham_data, wave_data, t1)
o_ex, e_ex = trial._calc_energy_exp_xtau(walker_up, walker_dn, ham_data, wave_data, t1)
t2_aa = jnp.einsum('ia,jb->iajb',t1[0],t1[0])
t2_ab = jnp.einsum('ia,jb->iajb',t1[0],t1[1])
t2_bb = jnp.einsum('ia,jb->iajb',t1[1],t1[1])
t2 = [t2_aa, t2_ab, t2_bb]
print(mf.e_tot + mycc.energy(t1, t2))
print(e_ci)
print(e_ex)

-1.0001123556227587
(-0.9992999164291994+0j)
(-1.000126077362184+0j)


In [13]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape))
walker_dn = jnp.array(np.random.rand(*walker_dn.shape))
norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
ci2 = [wave_data['t2aa'], wave_data['t2ab'], wave_data['t2bb']] 
olp, energy = _calc_energy_ad(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
print(olp, energy-mycc.e_tot)

1.3058148502829299 -2.240568117706232e-07


In [75]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape) + 1j * np.random.rand(*walker_up.shape)) * 0.5
walker_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j * np.random.rand(*walker_dn.shape)) * 0.5
slater_up = jnp.array(np.random.rand(*walker_up.shape) + 1j*np.random.rand(*walker_up.shape)) * 0.5
slater_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j*np.random.rand(*walker_dn.shape)) * 0.5
wave_data['mo_ta'] = slater_up
wave_data['mo_tb'] = slater_dn

norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
ci2 = [wave_data['t2aa'], wave_data['t2ab'], wave_data['t2bb']] 
olp1, energy1 = _calc_energy_ad(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
olp2, energy2 = _calc_energy_cid(trial, walker_up, walker_dn, ham_data, wave_data, ci2)
print(olp1, energy1)
print(olp2, energy2)

(0.4693251761161401+0.1389904858527459j) (-0.4775885034613132-0.14297453250648984j)
(0.4693251761161401+0.1389904858527459j) (-0.4775885034613131-0.14297453250648978j)


In [97]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape) + 1j * np.random.rand(*walker_up.shape)) * 0.5
walker_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j * np.random.rand(*walker_dn.shape)) * 0.5
norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
# ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
ci1a = jnp.array(np.random.rand(nocca, nvira) + 1j*np.random.rand(nocca, nvira))
ci1b = jnp.array(np.random.rand(noccb, nvirb) + 1j*np.random.rand(noccb, nvirb))
ci2aa = jnp.einsum('ia,jb->iajb', ci1a, ci1a)
ci2ab = jnp.einsum('ia,jb->iajb', ci1a, ci1b)
ci2bb = jnp.einsum('ia,jb->iajb', ci1b, ci1b)
ci1 = [ci1a, ci1b]
# ciaa = jnp.array(np.random.rand(*wave_data['t2aa'].shape) + 1j*np.random.rand(*wave_data['t2aa'].shape)) * 0.5
ci2 = [wave_data['t2aa'], wave_data['t2ab'], wave_data['t2bb']]
olp1, energy1 = _calc_energy_ad(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
print(olp1, energy1)
# olp2, energy2 = _calc_energy_cid(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
# print(olp2, energy2)
olp3, energy3 = _calc_energy_cisd(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
print(olp3, energy3)

(0.8751727603901351-0.548150123657969j) (-0.2075504047784511-0.2798992836208454j)
(0.8751727603901351-0.5481501236579691j) (-0.20755040449070955-0.27989928402336406j)


In [102]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape) + 1j * np.random.rand(*walker_up.shape)) * 0.5
walker_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j * np.random.rand(*walker_dn.shape)) * 0.5
slater_up = jnp.array(np.random.rand(*walker_up.shape) + 1j*np.random.rand(*walker_up.shape)) * 0.5
slater_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j*np.random.rand(*walker_dn.shape)) * 0.5
wave_data['mo_ta'] = slater_up
wave_data['mo_tb'] = slater_dn
norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
# ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
ci1a = jnp.array(np.random.rand(nocca, nvira) + 1j*np.random.rand(nocca, nvira)) * 0.5
ci1b = jnp.array(np.random.rand(noccb, nvirb) + 1j*np.random.rand(noccb, nvirb)) * 0.5
ci2aa = jnp.einsum('ia,jb->iajb', ci1a, ci1a)
ci2ab = jnp.einsum('ia,jb->iajb', ci1a, ci1b)
ci2bb = jnp.einsum('ia,jb->iajb', ci1b, ci1b)
ci1 = [ci1a, ci1b]
# ciaa = jnp.array(np.random.rand(*wave_data['t2aa'].shape) + 1j*np.random.rand(*wave_data['t2aa'].shape)) * 0.5
ci2 = [ci2aa, ci2ab, ci2bb]
olp1, energy1 = _calc_energy_ad(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
print(olp1, energy1)
# olp2, energy2 = _calc_energy_cid(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
# print(olp2, energy2)
olp3, energy3 = _calc_energy_cisd(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
print(olp3, energy3)
olp4, energy4 = _calc_energy_cisd_dc(trial, walker_up, walker_dn, ham_data, wave_data, ci1)
print(olp4, energy4)

(0.3090011489922681-0.11982103000821652j) (0.0688251638440352+0.24555066933502867j)
(0.3090011489922681-0.11982103000821652j) (0.06882516310995715+0.24555066750832955j)
(0.3090011489922681-0.11982103000821655j) (0.06882516384403545+0.245550669335029j)
